In [1]:
import pandas as pd
from load_data import load_data

In [2]:
from load_data import load_data
from preprocess import add_bag_of_words_column
from frequency import get_wordcounts
from matching import compute_tfidf_similarity, count_matching_keywords_no_repeats, encoder_scoring

import os
from recommendation import recommend_skills
import pandas as pd
from sentence_transformers import SentenceTransformer

/Users/luis/.pyenv/versions/3.12.9/envs/jobable2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
job_df = load_data('../data/job_title_des.csv')
resume_df = load_data('../data/gpt_dataset.csv')

#select resume and job description instance from df
test_job_instance = job_df.iloc[9, 2] #change first number to change resume instance (e.g [10,2] for the resume in index 10)

test_resume_instance = resume_df.iloc[34,1] #change first number to change resume instance (e.g [10,1] for the resume in index 10)
# test_resume_instance = resume_df[resume_df['Category']=='ENGINEERING'].iloc[5,1]

#use category=engineering to get resumes with datascience keywords
# test_resume_id = resume_df[resume_df['Resume_str']==test_resume_instance].iloc[0,0]

#use data from gpt_dataset
test_resume_id = resume_df[resume_df['Resume']==test_resume_instance].iloc[0,0]

In [16]:
test_resume_id

'Mobile App Developer (iOS/Android)'

In [11]:
job_df

,Unnamed: 0,Job Title,Job Description,tfidf_score
0,0,Flutter Developer,We are looking for hire experts flutter develo...,0.036614
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,0.243681
2,2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n...",0.372346
3,3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...,0.473227
4,4,Full Stack Developer,job responsibility full stack engineer – react...,0.044319
...,...,...,...,...
2272,2399,Backend Developer,Job Summary\nPublished on : 26 days ago\nVacan...,0.374675
2273,2400,Full Stack Developer,business entity cisco umbrella focus cloud-bas...,0.021144
2274,2401,Network Administrator,Urgently reqd in a college in Mohali\nNetwork ...,0.055815
2275,2402,Machine Learning,Key Responsibilities: Team leads for small or ...,0.314349


In [19]:
test_encoder_scoring(test_job_instance, test_resume_instance)

array([0.3112855], dtype=float32)

In [4]:
def test_tf_idf(test_job=test_job_instance, test_resume=test_resume_instance):
    return compute_tfidf_similarity(test_job, test_resume)

def test_count_matching_keywords_no_repeats(test_job=test_job_instance, test_resume=test_resume_instance):
    return count_matching_keywords_no_repeats(test_job, test_resume)

def test_encoder_scoring(test_job=test_job_instance, test_resume=test_resume_instance, model=None):
    return encoder_scoring(test_job, test_resume, model)

In [ ]:
def test_all_scoring_functions(test_resume=test_resume_instance, job_df=job_df, save_csv=True):

    top10_idf = pd.DataFrame()
    top10_encoder = pd.DataFrame()


    #TFIDF
    job_df['tfidf_score'] = job_df['Job Description'].apply(lambda x: test_tf_idf(x, test_resume))
    t = job_df.sort_values(by='tfidf_score', ascending=False).head(50) #higher is better

    top10_idf['idf_job_index'] = t.index
    top10_idf[['idf_job_descriptions', 'tfidf_score']] = t[['Job Description', 'tfidf_score']].values
    #KEYWORD MATCHING
    #job_df['matching_score'] = job_df['Job Description'].apply(lambda x: test_count_matching_keywords_no_repeats(x, test_resume))
    #t = job_df.sort_values(by='matching_score', ascending=False).head(10)

    #top10_df['keyword_matching_job_index'] = t.index
    #top10_df[['keyword_matching_job_descriptions', 'keyword_matching_scores']] = t[['Job Description','matching_score']].values

    print('M: finished keyword matching tests')



    #ENCODER
    model = SentenceTransformer("all-MiniLM-L6-v2")

    ### GET THE EMBEDDINGS
    embeddings = model.encode(job_df['Job Description'][0])

    print("M:finished computing the embeddings")

    job_df['encoder_scores'] = job_df['Job Description'].apply(lambda x: test_encoder_scoring(x, test_resume, model))
    t= job_df.sort_values(by='encoder_scores', ascending=False).head(50)

    top10_encoder['encoder_job_index'] = t.index
    top10_encoder[['encoder_job_descriptions', 'encoder_scores']] = t[['Job Description', 'encoder_scores']].values
    print('M: finished keyword encoder tests')


    return top10_idf, top10_encoder

In [119]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(job_df['Job Description'])
embeddings

array([[-0.06900153, -0.01736592,  0.04985921, ..., -0.06563059,
         0.03301352,  0.07066134],
       [-0.08982832, -0.03184036, -0.03589954, ...,  0.05279462,
         0.0590908 ,  0.07545409],
       [-0.10459907, -0.03273258,  0.00064838, ..., -0.02026048,
        -0.07207101, -0.00901509],
       ...,
       [-0.04923702, -0.00656475,  0.00596427, ..., -0.08362456,
        -0.08244597, -0.03827264],
       [-0.03547642,  0.00449758, -0.00860993, ..., -0.01648797,
        -0.02395136,  0.04224841],
       [-0.01959286, -0.04775559, -0.01091558, ..., -0.10431547,
        -0.07700158,  0.10585925]], dtype=float32)

In [ ]:
job_df['embeddings'] = embeddings
e

ValueError: Expected a 1D array, got an array with shape (2277, 384)

In [127]:
def embed_to_column(row):
    row_indx = row.name
    row['embeddings'] = embeddings[row_indx]
    return row

In [130]:
job_df = job_df.apply(embed_to_column, axis = 1)

384

In [114]:
embeddings_df = pd.DataFrame(embeddings[0], columns=['Values'])
embeddings_df

,Values
0,-0.069002
1,-0.017366
2,0.049859
3,-0.006420
4,-0.015447
...,...
379,-0.002939
380,0.030722
381,-0.065631
382,0.033014


In [ ]:
job_df['embeddings'] = job_df['Job Description']

ValueError: Length of values (384) does not match length of index (2277)

In [95]:
t_idf, t_encoder = test_all_scoring_functions(test_resume=test_resume_instance, job_df=job_df, save_csv=True)
model = SentenceTransformer("all-MiniLM-L6-v2")


M: finished keyword matching tests
M: finished keyword encoder tests


In [96]:
t_idf

,idf_job_index,idf_job_descriptions,tfidf_score
0,1508,Who we are: Fueled by a fundamental belief tha...,0.537097
1,1790,Software Developer - Web\n2200 S. Lakeside Dri...,0.536967
2,2057,Company Description\nPositioned at Publicis Gr...,0.536853
3,1309,A professional iphone app developer who can le...,0.533794
4,509,"Responsible for design, development and implem...",0.530229
5,1163,Who we are: Fueled by a fundamental belief tha...,0.528946
6,2133,"Position Description: As a DevOps Engineer, yo...",0.522699
7,1661,Areas of responsibility: -\nThe candidate will...,0.521835
8,588,Areas of responsibility: -\nThe candidate will...,0.521712
9,1329,Job Responsibilities\nDesign and build advance...,0.521667


In [97]:
t_encoder

,encoder_job_index,encoder_job_descriptions,encoder_scores
0,1145,Requirements:\nCross-platform app developers w...,[0.72247183]
1,1329,Job Responsibilities\nDesign and build advance...,[0.701586]
2,1854,Roles and responsibilities: *\nCross-platform ...,[0.6955334]
3,53,The iOS Swift Developer will be an individual ...,[0.6819308]
4,1129,Required Skill:\nProven experience in software...,[0.68024683]
5,822,Job Summary\n5+ years’ experience in developin...,[0.6791451]
6,1309,A professional iphone app developer who can le...,[0.67779326]
7,717,Flutter Developer Job Duties: *\n· Ability to ...,[0.67398846]
8,1595,Job Description*\n2-5 years of Mobile app deve...,[0.66814154]
9,1186,Work Experience: 3-5 years\nLocation: Gurugram...,[0.6517352]


In [51]:
job_df.iloc[711,:]

Unnamed: 0                                                       730
Job Title                                          Flutter Developer
Job Description    FullThrottle Labs is looking for an Mobile App...
tfidf_score                                                  0.50572
Name: 711, dtype: object

In [34]:
top10_df = pd.DataFrame()


In [36]:
top10_df['encoder_job_index'] = t.index
top10_df[['encoder_job_descriptions', 'encoder_scores']] = t[['Job Description','encoder_scores']].values
print('M: finished keyword encoder tests')
top10_df

M: finished keyword encoder tests


,encoder_job_index,encoder_job_descriptions,encoder_scores
0,711,FullThrottle Labs is looking for an Mobile App...,[0.108134486]
1,181,Design and Build sophisticated and highly scal...,[0.09719333]
2,2186,Design and Build sophisticated and highly scal...,[0.094065845]
3,892,STP provides Fund Administration and Performan...,[0.07479511]
4,51,Job Description :\nHitachi Vantara combines te...,[0.071972474]
5,198,"iPhone Developer:\n\nTo excel at Simform, you ...",[0.0695595]
6,588,Areas of responsibility: -\nThe candidate will...,[0.06488245]
7,2162,We are hiring for one of the top IT company in...,[0.06426556]
8,2155,About the company*\nBlackbean Marketing Techno...,[0.06297637]
9,778,"We are seeking an iOS Developer, iOS to be res...",[0.061980546]


In [ ]:
pd.read_csv('../data/job_title_des.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../../data/job_title_des.csv'

In [ ]:
job_df = load_data('../data/job_title_des.csv')
resume_df = load_data('../data/job_title_des.csv')

#select resume and job instance from df
test_job_instance = job_df.iloc[9, 2] #change first number to change resume instance (e.g [10,2] for the resume in index 10)

test_resume_instance = resume_df.iloc[5,1] #change first number to change resume instance (e.g [10,1] for the resume in index 10)
# test_resume_instance = resume_df[resume_df['Category']=='ENGINEERING'].iloc[5,1]
print(resume_df.columns)
#use category=engineering to get resumes with datascience keywords

test_resume_id = resume_df[resume_df['Resume_str']==test_resume_instance].iloc[0,0]


def test_tf_idf(test_job=test_job_instance, test_resume=test_resume_instance):
    return compute_tfidf_similarity(test_job, test_resume)

def test_count_matching_keywords_no_repeats(test_job=test_job_instance, test_resume=test_resume_instance):
    return count_matching_keywords_no_repeats(test_job, test_resume)

def test_encoder_scoring(test_job=test_job_instance, test_resume=test_resume_instance, model=None):
    return encoder_scoring(test_job, test_resume, model)
